# DocLite: Knowledge Distillation for Document Understanding

**Teacher:** LayoutLMv3 (text + layout + image) → **Student:** LiLT (text + layout only)

This notebook runs all experiments:
1. LayoutLMv3 supervised baseline (teacher upper bound)
2. LiLT supervised baseline (student lower bound)
3. DocLite distillation (our contribution)
4. Multimodal distillation with images

On both **FUNSD** (forms) and **SROIE** (receipts) datasets.

## 0. Setup

In [ ]:
!git clone https://github.com/lukeengel/doclite.git
%cd doclite
!pip install -r requirements.txt -q
!apt-get install -y tesseract-ocr -qq

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Upload SROIE dataset (optional)

If you have SROIE, zip it and upload to Google Drive, then uncomment and run the cell below.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/sroie.zip -d data/

## 1. Imports & Config

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, LayoutLMv3Config, LayoutLMv3ForTokenClassification
from collections import Counter

from doclite.configs.core import ENV, WEIGHTS
from doclite.data.document_dataset import DocumentDataset
from doclite.data.collate import collate_document_batch
from doclite.eval.evaluate import evaluate_student
from doclite.models.teacher import TeacherModel
from doclite.models.student import StudentModel
from doclite.distill.distill_loss import compute_distill_loss
from build_funsd_examples import load_funsd_split

device = "cuda" if torch.cuda.is_available() else "cpu"

# Eval wrapper reused across experiments
class LayoutLMv3EvalWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_ids, attention_mask, bbox, **kwargs):
        fwd = dict(input_ids=input_ids, attention_mask=attention_mask, bbox=bbox)
        if 'pixel_values' in kwargs:
            fwd['pixel_values'] = kwargs['pixel_values']
        return {"logits": self.model(**fwd).logits}

# Hyperparameters
BATCH_SIZE = 8
NUM_EPOCHS = 5
LR = 5e-5
NUM_LABELS_FUNSD = 4

# Store all results here
all_results = {}

print(f"Device: {device} | Batch: {BATCH_SIZE} | Epochs: {NUM_EPOCHS} | LR: {LR}")

---
# PART 1: FUNSD Experiments (Text + Layout)
---

## 2. Load FUNSD Data

In [ ]:
FUNSD_ROOT = ENV.DATA / "funsd"
TRAIN_ANN_DIR = FUNSD_ROOT / "training_data" / "annotations"
TEST_ANN_DIR = FUNSD_ROOT / "testing_data" / "annotations"

tokenizer = AutoTokenizer.from_pretrained("microsoft/layoutlmv3-base")

funsd_train = load_funsd_split(TRAIN_ANN_DIR, tokenizer)
funsd_test = load_funsd_split(TEST_ANN_DIR, tokenizer)

funsd_train_loader = DataLoader(
    DocumentDataset(funsd_train), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_document_batch
)
funsd_test_loader = DataLoader(
    DocumentDataset(funsd_test), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_document_batch
)

print(f"FUNSD — Train: {len(funsd_train)} docs ({len(funsd_train_loader)} batches) | Test: {len(funsd_test)} docs")

ID2LABEL_FUNSD = {0: "question", 1: "answer", 2: "header", 3: "other", -100: "[PAD]"}
all_labels = [l for ex in funsd_train for l in ex["labels"] if l != -100]
counts = Counter(all_labels)
print("\nLabel distribution (training):")
for lid, c in sorted(counts.items()):
    print(f"  {ID2LABEL_FUNSD[lid]:>10s}: {c:>5d} ({c/len(all_labels)*100:.1f}%)")

## 3. Experiment A: LayoutLMv3 Baseline (Teacher)

In [ ]:
config = LayoutLMv3Config.from_pretrained(
    "microsoft/layoutlmv3-base", num_labels=NUM_LABELS_FUNSD,
    output_hidden_states=True, output_attentions=True,
)
model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base", config=config
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
eval_wrapper = LayoutLMv3EvalWrapper(model).to(device)

all_results["teacher_params"] = sum(p.numel() for p in model.parameters())
print(f"LayoutLMv3 parameters: {all_results['teacher_params']:,}")

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(funsd_train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                    bbox=batch["bbox"], labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
        if step % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(funsd_train_loader)} | loss={out.loss.item():.4f}")
    metrics = evaluate_student(eval_wrapper, funsd_test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1} | train_loss={total_loss/len(funsd_train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_teacher"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_teacher']['token_f1']:.4f}")

# Free GPU memory
del model, optimizer, eval_wrapper
torch.cuda.empty_cache()

## 4. Experiment B: LiLT Baseline (Student)

In [ ]:
model = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS_FUNSD).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

all_results["student_params"] = sum(p.numel() for p in model.parameters())
print(f"LiLT parameters: {all_results['student_params']:,}")

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(funsd_train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model.model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                         bbox=batch["bbox"], labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
        if step % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(funsd_train_loader)} | loss={out.loss.item():.4f}")
    metrics = evaluate_student(model, funsd_test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1} | train_loss={total_loss/len(funsd_train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_student"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_student']['token_f1']:.4f}")

del model, optimizer
torch.cuda.empty_cache()

## 5. Experiment C: DocLite Distillation

In [ ]:
teacher = TeacherModel("microsoft/layoutlmv3-base", num_labels=NUM_LABELS_FUNSD).to(device)
for param in teacher.parameters():
    param.requires_grad = False

student = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS_FUNSD).to(device)
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)

print(f"Weights: ALPHA={WEIGHTS.ALPHA_HIDDEN}, BETA={WEIGHTS.BETA_ATTN}, GAMMA={WEIGHTS.GAMMA_LOGITS}, DELTA={WEIGHTS.DELTA_TASK}")

results = []
for epoch in range(NUM_EPOCHS):
    teacher.eval()
    student.train()
    total_loss = 0.0
    for step, batch in enumerate(funsd_train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        model_inputs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"], "bbox": batch["bbox"]}
        teacher_out = teacher(**model_inputs)

        student_hf_out = student.model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], labels=batch["labels"],
            output_hidden_states=True, output_attentions=True, return_dict=True,
        )
        student_out = {"logits": student_hf_out.logits, "hidden_states": student_hf_out.hidden_states, "attentions": student_hf_out.attentions}

        distill_losses = compute_distill_loss(teacher_out, student_out, attention_mask=batch["attention_mask"])
        loss = distill_losses["loss_total"] + WEIGHTS.DELTA_TASK * student_hf_out.loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if step % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(funsd_train_loader)} | loss={loss.item():.4f}")

    metrics = evaluate_student(student, funsd_test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1} | train_loss={total_loss/len(funsd_train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_distill"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_distill']['token_f1']:.4f}")

del teacher, student, optimizer
torch.cuda.empty_cache()

## 6. FUNSD Results (Text + Layout)

In [ ]:
t = all_results["funsd_teacher"]
s = all_results["funsd_student"]
d = all_results["funsd_distill"]
tp = all_results["teacher_params"]
sp = all_results["student_params"]

print("=" * 70)
print("FUNSD RESULTS (Text + Layout)")
print("=" * 70)
print(f"{'Model':<30s} {'Accuracy':>10s} {'F1':>10s} {'Params':>12s}")
print("-" * 70)
print(f"{'LayoutLMv3 (teacher)':<30s} {t['token_acc']:>10.4f} {t['token_f1']:>10.4f} {tp:>12,}")
print(f"{'LiLT (baseline)':<30s} {s['token_acc']:>10.4f} {s['token_f1']:>10.4f} {sp:>12,}")
print(f"{'LiLT + DocLite (ours)':<30s} {d['token_acc']:>10.4f} {d['token_f1']:>10.4f} {sp:>12,}")
print("=" * 70)
gap = (d['token_f1'] - s['token_f1']) / (t['token_f1'] - s['token_f1'] + 1e-8) * 100
print(f"\nDistillation closed {gap:.1f}% of the teacher-student F1 gap")
print(f"Compression: {tp/sp:.1f}x fewer parameters")

---
# PART 2: Multimodal Distillation (Teacher sees images)

Teacher (LayoutLMv3) sees **text + layout + image**.  
Student (LiLT) sees only **text + layout**.  
The student learns visual knowledge **without ever seeing images**.

---

In [ ]:
# Load FUNSD WITH images
TRAIN_IMG_DIR = FUNSD_ROOT / "training_data" / "images"
TEST_IMG_DIR = FUNSD_ROOT / "testing_data" / "images"

print("Loading training data with images...")
funsd_train_img = load_funsd_split(TRAIN_ANN_DIR, tokenizer, image_dir=TRAIN_IMG_DIR)
print("Loading test data with images...")
funsd_test_img = load_funsd_split(TEST_ANN_DIR, tokenizer, image_dir=TEST_IMG_DIR)

funsd_train_img_loader = DataLoader(
    DocumentDataset(funsd_train_img), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_document_batch
)
funsd_test_img_loader = DataLoader(
    DocumentDataset(funsd_test_img), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_document_batch
)

print(f"Has pixel_values: {'pixel_values' in funsd_train_img[0]}")
print(f"pixel_values shape: {torch.tensor(funsd_train_img[0]['pixel_values']).shape}")

## 7. LayoutLMv3 + Image Baseline

In [ ]:
config = LayoutLMv3Config.from_pretrained(
    "microsoft/layoutlmv3-base", num_labels=NUM_LABELS_FUNSD,
    output_hidden_states=True, output_attentions=True,
)
model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base", config=config
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
eval_wrapper = LayoutLMv3EvalWrapper(model).to(device)

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(funsd_train_img_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                    bbox=batch["bbox"], pixel_values=batch.get("pixel_values"), labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
        if step % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(funsd_train_img_loader)} | loss={out.loss.item():.4f}")
    metrics = evaluate_student(eval_wrapper, funsd_test_img_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1} | train_loss={total_loss/len(funsd_train_img_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_teacher_img"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_teacher_img']['token_f1']:.4f}")

del model, optimizer, eval_wrapper
torch.cuda.empty_cache()

## 8. Multimodal DocLite Distillation

In [ ]:
teacher = TeacherModel("microsoft/layoutlmv3-base", num_labels=NUM_LABELS_FUNSD).to(device)
for param in teacher.parameters():
    param.requires_grad = False

student = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS_FUNSD).to(device)
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)

results = []
for epoch in range(NUM_EPOCHS):
    teacher.eval()
    student.train()
    total_loss = 0.0
    for step, batch in enumerate(funsd_train_img_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        # Teacher sees text + layout + IMAGE
        teacher_inputs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"], "bbox": batch["bbox"]}
        if "pixel_values" in batch:
            teacher_inputs["pixel_values"] = batch["pixel_values"]
        teacher_out = teacher(**teacher_inputs)

        # Student sees ONLY text + layout
        student_hf_out = student.model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], labels=batch["labels"],
            output_hidden_states=True, output_attentions=True, return_dict=True,
        )
        student_out = {"logits": student_hf_out.logits, "hidden_states": student_hf_out.hidden_states, "attentions": student_hf_out.attentions}

        distill_losses = compute_distill_loss(teacher_out, student_out, attention_mask=batch["attention_mask"])
        loss = distill_losses["loss_total"] + WEIGHTS.DELTA_TASK * student_hf_out.loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if step % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(funsd_train_img_loader)} | loss={loss.item():.4f}")

    metrics = evaluate_student(student, funsd_test_img_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1} | train_loss={total_loss/len(funsd_train_img_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_mm_distill"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_mm_distill']['token_f1']:.4f}")

del teacher, student, optimizer
torch.cuda.empty_cache()

---
# PART 3: SROIE Experiments (Receipts)

Skip this section if you haven't uploaded the SROIE dataset.

---

In [ ]:
from build_sroie_examples import load_sroie_split, NUM_LABELS as NUM_LABELS_SROIE

SROIE_ROOT = ENV.DATA / "sroie"
TRAIN_OCR_DIR = SROIE_ROOT / "train" / "box"
TRAIN_ENT_DIR = SROIE_ROOT / "train" / "entities"
TEST_OCR_DIR = SROIE_ROOT / "test" / "box"
TEST_ENT_DIR = SROIE_ROOT / "test" / "entities"

sroie_train = load_sroie_split(TRAIN_OCR_DIR, TRAIN_ENT_DIR, tokenizer)
sroie_test = load_sroie_split(TEST_OCR_DIR, TEST_ENT_DIR, tokenizer)

sroie_train_loader = DataLoader(
    DocumentDataset(sroie_train), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_document_batch
)
sroie_test_loader = DataLoader(
    DocumentDataset(sroie_test), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_document_batch
)

print(f"SROIE — Train: {len(sroie_train)} receipts | Test: {len(sroie_test)} receipts")

ID2LABEL_SROIE = {0: "company", 1: "date", 2: "address", 3: "total", 4: "other", -100: "[PAD]"}
all_labels_s = [l for ex in sroie_train for l in ex["labels"] if l != -100]
counts_s = Counter(all_labels_s)
print("\nLabel distribution (training):")
for lid, c in sorted(counts_s.items()):
    print(f"  {ID2LABEL_SROIE[lid]:>10s}: {c:>5d} ({c/len(all_labels_s)*100:.1f}%)")

## 9. SROIE: Teacher Baseline

In [ ]:
config = LayoutLMv3Config.from_pretrained(
    "microsoft/layoutlmv3-base", num_labels=NUM_LABELS_SROIE,
    output_hidden_states=True, output_attentions=True,
)
model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base", config=config
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
eval_wrapper = LayoutLMv3EvalWrapper(model).to(device)

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(sroie_train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                    bbox=batch["bbox"], labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    metrics = evaluate_student(eval_wrapper, sroie_test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1} | train_loss={total_loss/len(sroie_train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["sroie_teacher"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['sroie_teacher']['token_f1']:.4f}")

del model, optimizer, eval_wrapper
torch.cuda.empty_cache()

## 10. SROIE: Student Baseline

In [ ]:
model = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS_SROIE).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(sroie_train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model.model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                         bbox=batch["bbox"], labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    metrics = evaluate_student(model, sroie_test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1} | train_loss={total_loss/len(sroie_train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["sroie_student"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['sroie_student']['token_f1']:.4f}")

del model, optimizer
torch.cuda.empty_cache()

## 11. SROIE: DocLite Distillation

In [ ]:
teacher = TeacherModel("microsoft/layoutlmv3-base", num_labels=NUM_LABELS_SROIE).to(device)
for param in teacher.parameters():
    param.requires_grad = False

student = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS_SROIE).to(device)
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)

results = []
for epoch in range(NUM_EPOCHS):
    teacher.eval()
    student.train()
    total_loss = 0.0
    for step, batch in enumerate(sroie_train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        model_inputs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"], "bbox": batch["bbox"]}
        teacher_out = teacher(**model_inputs)

        student_hf_out = student.model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], labels=batch["labels"],
            output_hidden_states=True, output_attentions=True, return_dict=True,
        )
        student_out = {"logits": student_hf_out.logits, "hidden_states": student_hf_out.hidden_states, "attentions": student_hf_out.attentions}

        distill_losses = compute_distill_loss(teacher_out, student_out, attention_mask=batch["attention_mask"])
        loss = distill_losses["loss_total"] + WEIGHTS.DELTA_TASK * student_hf_out.loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    metrics = evaluate_student(student, sroie_test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1} | train_loss={total_loss/len(sroie_train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["sroie_distill"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['sroie_distill']['token_f1']:.4f}")

del teacher, student, optimizer
torch.cuda.empty_cache()

---
# PART 4: Final Summary
---

In [ ]:
tp = all_results["teacher_params"]
sp = all_results["student_params"]

print("=" * 80)
print("COMPLETE RESULTS SUMMARY")
print("=" * 80)

# FUNSD
print(f"\n{'FUNSD (Forms — 4 labels)':^80s}")
print("-" * 80)
print(f"{'Model':<35s} {'Inputs':<12s} {'Acc':>8s} {'F1':>8s} {'Params':>12s}")
print("-" * 80)

ft = all_results["funsd_teacher"]
fs = all_results["funsd_student"]
fd = all_results["funsd_distill"]
print(f"{'LayoutLMv3 (teacher)':<35s} {'T+L':<12s} {ft['token_acc']:>8.4f} {ft['token_f1']:>8.4f} {tp:>12,}")
print(f"{'LiLT (baseline)':<35s} {'T+L':<12s} {fs['token_acc']:>8.4f} {fs['token_f1']:>8.4f} {sp:>12,}")
print(f"{'LiLT + DocLite (ours)':<35s} {'T+L':<12s} {fd['token_acc']:>8.4f} {fd['token_f1']:>8.4f} {sp:>12,}")

if "funsd_teacher_img" in all_results:
    fti = all_results["funsd_teacher_img"]
    fmm = all_results["funsd_mm_distill"]
    print(f"{'LayoutLMv3 + Image (teacher)':<35s} {'T+L+I':<12s} {fti['token_acc']:>8.4f} {fti['token_f1']:>8.4f} {tp:>12,}")
    print(f"{'LiLT + DocLite multimodal':<35s} {'T+L':<12s} {fmm['token_acc']:>8.4f} {fmm['token_f1']:>8.4f} {sp:>12,}")

gap = (fd['token_f1'] - fs['token_f1']) / (ft['token_f1'] - fs['token_f1'] + 1e-8) * 100
print(f"\n  Distillation closed {gap:.1f}% of the teacher-student F1 gap")

# SROIE
if "sroie_teacher" in all_results:
    print(f"\n{'SROIE (Receipts — 5 labels)':^80s}")
    print("-" * 80)
    print(f"{'Model':<35s} {'Inputs':<12s} {'Acc':>8s} {'F1':>8s}")
    print("-" * 80)
    st = all_results["sroie_teacher"]
    ss = all_results["sroie_student"]
    sd = all_results["sroie_distill"]
    print(f"{'LayoutLMv3 (teacher)':<35s} {'T+L':<12s} {st['token_acc']:>8.4f} {st['token_f1']:>8.4f}")
    print(f"{'LiLT (baseline)':<35s} {'T+L':<12s} {ss['token_acc']:>8.4f} {ss['token_f1']:>8.4f}")
    print(f"{'LiLT + DocLite (ours)':<35s} {'T+L':<12s} {sd['token_acc']:>8.4f} {sd['token_f1']:>8.4f}")
    gap_s = (sd['token_f1'] - ss['token_f1']) / (st['token_f1'] - ss['token_f1'] + 1e-8) * 100
    print(f"\n  Distillation closed {gap_s:.1f}% of the teacher-student F1 gap")

print("\n" + "=" * 80)
print(f"Key: T=Text, L=Layout, I=Image")
print(f"Compression: {tp/sp:.1f}x fewer parameters")
print(f"Multimodal student NEVER sees images — learns visual knowledge through distillation.")
print("=" * 80)